In [1]:
import SimpleITK as sitk
import numpy as np
import torch
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import os
import glob
from pathlib import Path
import configparser
import shutil

In [6]:
nii_path = "datasets/oasis3_T1w_reformated/images"  # path to the nii images
#gt_path = "datasets/CAMUS_reformated/labels"  # path to the ground truth

modality = "MR_T1w"
anatomy = "Brain"  # anantomy + dataset name
img_name_suffix = ".nii.gz"
#gt_name_suffix = "_gt.nii.gz"
prefix = modality + "_" + anatomy + "_"


names = sorted(os.listdir(gt_path))
print(f"ori \# files {len(names)=}")
names = [
    name
    for name in names
    if os.path.exists(os.path.join(nii_path, name.split(gt_name_suffix)[0] + img_name_suffix))
]
print(f"after sanity check \# files {len(names)=}")

ori \# files len(names)=2000
after sanity check \# files len(names)=2000


## medSAM ACDC, CAMUS preprocessing

In [10]:
def ensure_3d(arr):
    """
    Ensure array is 3D in (Z, Y, X) format.
    If 2D (Y, X), add a singleton Z dimension.
    """
    if arr.ndim == 2:
        arr = arr[np.newaxis, ...]  # shape -> (1, Y, X)
    return arr

In [11]:
# -*- coding: utf-8 -*-
# %% import packages
# pip install connected-components-3d
import numpy as np

# import nibabel as nib
import SimpleITK as sitk
import os

join = os.path.join
from skimage import transform
from tqdm import tqdm
import cc3d

# convert nii image to npz files, including original image and corresponding masks
dataset_prefix="OASIS3"
modality = "MR_T1w"
anatomy = "Brain"  # anantomy + dataset name
img_name_suffix = ".nii.gz"
#gt_name_suffix = "_gt.nii.gz"
prefix = dataset_prefix + "_" + modality + "_" + anatomy + "_"

nii_path = "datasets/oasis3_T1w_reformated/images"  # path to the nii images
#gt_path = "datasets/CAMUS_reformated/labels"
npy_path = "datasets/prep_datasets/" + prefix[:-1]
#os.makedirs(join(npy_path, "gts"), exist_ok=True)
os.makedirs(join(npy_path, "imgs"), exist_ok=True)

image_size = 1024
voxel_num_thre2d = 100
voxel_num_thre3d = 1000

names = sorted(os.listdir(nii_path))
print(f"ori \# files {len(names)=}")
names = [
    name
    for name in names
    if os.path.exists(join(nii_path, name.split(gt_name_suffix)[0] + img_name_suffix))
]
print(f"after sanity check \# files {len(names)=}")

# set label ids that are excluded
remove_label_ids = [] 
tumor_id = None  # only set this when there are multiple tumors; convert semantic masks to instance masks
# set window level and width
# https://radiopaedia.org/articles/windowing-ct
WINDOW_LEVEL = 40  # only for CT images
WINDOW_WIDTH = 400  # only for CT images

# %% save preprocessed images and masks as npz files
for name in tqdm(names): 
#     image_name = name.split(gt_name_suffix)[0] + img_name_suffix
#     gt_name = name
#     gt_sitk = sitk.ReadImage(join(gt_path, gt_name))
#     gt_data_ori = np.uint8(sitk.GetArrayFromImage(gt_sitk))
#     gt_data_ori = ensure_3d(gt_data_ori)
#     # remove label ids
#     for remove_label_id in remove_label_ids:
#         gt_data_ori[gt_data_ori == remove_label_id] = 0
#     # label tumor masks as instances and remove from gt_data_ori
#     if tumor_id is not None:
#         tumor_bw = np.uint8(gt_data_ori == tumor_id)
#         gt_data_ori[tumor_bw > 0] = 0
#         # label tumor masks as instances
#         tumor_inst, tumor_n = cc3d.connected_components(
#             tumor_bw, connectivity=26, return_N=True
#         )
#         # put the tumor instances back to gt_data_ori
#         gt_data_ori[tumor_inst > 0] = (
#             tumor_inst[tumor_inst > 0] + np.max(gt_data_ori) + 1
#         )

#     # exclude the objects with less than 1000 pixels in 3D
#     gt_data_ori = cc3d.dust(
#         gt_data_ori, threshold=voxel_num_thre3d, connectivity=26, in_place=True
#     )
#     # remove small objects with less than 100 pixels in 2D slices

#     for slice_i in range(gt_data_ori.shape[0]):
#         gt_i = gt_data_ori[slice_i, :, :]
#         # remove small objects with less than 100 pixels
#         # reason: fro such small objects, the main challenge is detection rather than segmentation
#         gt_data_ori[slice_i, :, :] = cc3d.dust(
#             gt_i, threshold=voxel_num_thre2d, connectivity=8, in_place=True
#         )
#     # find non-zero slices
#     z_index, _, _ = np.where(gt_data_ori > 0)
#     z_index = np.unique(z_index)

    # if len(z_index) > 0:
        # crop the ground truth with non-zero slices
        gt_roi = gt_data_ori[z_index, :, :]
        # load image and preprocess
        img_sitk = sitk.ReadImage(join(nii_path, image_name))
        image_data = sitk.GetArrayFromImage(img_sitk)
        image_data = ensure_3d(image_data)
        # nii preprocess start
        if modality == "CT":
            lower_bound = WINDOW_LEVEL - WINDOW_WIDTH / 2
            upper_bound = WINDOW_LEVEL + WINDOW_WIDTH / 2
            image_data_pre = np.clip(image_data, lower_bound, upper_bound)
            image_data_pre = (
                (image_data_pre - np.min(image_data_pre))
                / (np.max(image_data_pre) - np.min(image_data_pre))
                * 255.0
            )
        else:
            lower_bound, upper_bound = np.percentile(
                image_data[image_data > 0], 0.5
            ), np.percentile(image_data[image_data > 0], 99.5)
            image_data_pre = np.clip(image_data, lower_bound, upper_bound)
            image_data_pre = (
                (image_data_pre - np.min(image_data_pre))
                / (np.max(image_data_pre) - np.min(image_data_pre))
                * 255.0
            )
            image_data_pre[image_data == 0] = 0

        image_data_pre = np.uint8(image_data_pre)
        img_roi = image_data_pre[z_index, :, :]
        np.savez_compressed(join(npy_path, prefix + gt_name.split(gt_name_suffix)[0]+'.npz'), imgs=img_roi, gts=gt_roi, spacing=img_sitk.GetSpacing())
        # save the image and ground truth as nii files for sanity check;
        # they can be removed
        img_roi_sitk = sitk.GetImageFromArray(img_roi)
        gt_roi_sitk = sitk.GetImageFromArray(gt_roi)
        sitk.WriteImage(
            img_roi_sitk,
            join(npy_path, prefix + gt_name.split(gt_name_suffix)[0] + "_img.nii.gz"),
        )
        sitk.WriteImage(
            gt_roi_sitk,
            join(npy_path, prefix + gt_name.split(gt_name_suffix)[0] + "_gt.nii.gz"),
        )
        # save the each CT image as npy file
        for i in range(img_roi.shape[0]):
            img_i = img_roi[i, :, :]
            img_3c = np.repeat(img_i[:, :, None], 3, axis=-1)
            resize_img_skimg = transform.resize(
                img_3c,
                (image_size, image_size),
                order=3,
                preserve_range=True,
                mode="constant",
                anti_aliasing=True,
            )
            resize_img_skimg_01 = (resize_img_skimg - resize_img_skimg.min()) / np.clip(
                resize_img_skimg.max() - resize_img_skimg.min(), a_min=1e-8, a_max=None
            )  # normalize to [0, 1], (H, W, 3)
            gt_i = gt_roi[i, :, :]
            resize_gt_skimg = transform.resize(
                gt_i,
                (image_size, image_size),
                order=0,
                preserve_range=True,
                mode="constant",
                anti_aliasing=False,
            )
            resize_gt_skimg = np.uint8(resize_gt_skimg)
            assert resize_img_skimg_01.shape[:2] == resize_gt_skimg.shape
            np.save(
                join(
                    npy_path,
                    "imgs",
                    prefix
                    + gt_name.split(gt_name_suffix)[0]
                    + "-"
                    + str(i).zfill(3)
                    + ".npy",
                ),
                resize_img_skimg_01,
            )
            np.save(
                join(
                    npy_path,
                    "gts",
                    prefix
                    + gt_name.split(gt_name_suffix)[0]
                    + "-"
                    + str(i).zfill(3)
                    + ".npy",
                ),
                resize_gt_skimg,
            )


ori \# files len(names)=2000
after sanity check \# files len(names)=2000


  3%|▎         | 62/2000 [02:02<1:03:43,  1.97s/it]


KeyboardInterrupt: 

## ACDC class split preprocessing

In [44]:
dataset_path=Path("./datasets/ACDC/images")

for subdirectory in dataset_path.iterdir():
    if subdirectory.name not in ["training","testing"]:
        continue
        
    patients_path=dataset_path / subdirectory.name
    
    for patient_folder in list(patients_path.glob("patient*")):
        cfg_file=next(patient_folder.glob("*.cfg"), None)
        
        if cfg_file is None:
            continue
        
        info = {}
        with open(cfg_file) as f:
            for line in f:
                key, value = line.strip().split(":")
                info[key.strip()] = value.strip()

        if info["Group"] == "RV":
            dest=dataset_path / "ACDC_class_split" / "RV_class"
        else:
            dest=dataset_path / "ACDC_class_split" / "not_RV_class"
        
        shutil.copytree(patient_folder,dest / patient_folder.name)
            
            

In [2]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import SimpleITK as sitk
from skimage import transform
from tqdm import tqdm


def ensure_3d(arr):
    """Ensure array has shape (Z, H, W)."""
    if arr.ndim == 2:
        arr = arr[None, :, :]
    elif arr.ndim != 3:
        raise ValueError(f"Expected 2D or 3D image, got shape {arr.shape}")
    return arr


def preprocess_volume(image_data, modality="MR", window_level=40, window_width=400):
    """Normalize a 3D image volume to uint8 [0, 255]."""
    if modality == "CT":
        lower_bound = window_level - window_width / 2
        upper_bound = window_level + window_width / 2
        image_data_pre = np.clip(image_data, lower_bound, upper_bound)
    else:
        nonzero = image_data[image_data > 0]
        if nonzero.size == 0:
            image_data_pre = np.zeros_like(image_data, dtype=np.float32)
        else:
            lower_bound = np.percentile(nonzero, 0.5)
            upper_bound = np.percentile(nonzero, 99.5)
            image_data_pre = np.clip(image_data, lower_bound, upper_bound)

    min_val = image_data_pre.min()
    max_val = image_data_pre.max()

    if max_val > min_val:
        image_data_pre = (image_data_pre - min_val) / (max_val - min_val) * 255.0
    else:
        image_data_pre = np.zeros_like(image_data_pre, dtype=np.float32)

    if modality != "CT":
        image_data_pre[image_data == 0] = 0

    return np.uint8(image_data_pre)


def get_nonempty_slice_indices(volume):
    """Return slice indices whose content is not entirely zero."""
    keep = []
    for i in range(volume.shape[0]):
        if np.any(volume[i] > 0):
            keep.append(i)
    return keep


def process_one_volume(
    nii_file_str,
    input_root_str,
    output_root_str,
    prefix,
    modality,
    image_size,
    remove_empty_slices,
):
    nii_file = Path(nii_file_str)
    input_root = Path(input_root_str)
    output_root = Path(output_root_str)

    try:
        class_name = nii_file.relative_to(input_root).parts[0]
    except Exception:
        return f"Could not infer class for {nii_file}"

    if class_name not in {"RV_class", "not_RV_class"}:
        return f"Unexpected class folder for {nii_file}"

    try:
        img_sitk = sitk.ReadImage(str(nii_file))
        image_data = sitk.GetArrayFromImage(img_sitk)
        image_data = ensure_3d(image_data)

        image_data_pre = preprocess_volume(image_data, modality=modality)

        if remove_empty_slices:
            slice_indices = get_nonempty_slice_indices(image_data_pre)
        else:
            slice_indices = list(range(image_data_pre.shape[0]))

        if len(slice_indices) == 0:
            return f"No non-empty slices found in {nii_file}"

        volume_stem = nii_file.name.replace(".nii.gz", "")

        saved_count = 0
        for i in slice_indices:
            img_i = image_data_pre[i, :, :]

            # grayscale -> 3 channels
            img_3c = np.repeat(img_i[:, :, None], 3, axis=-1)

            resize_img = transform.resize(
                img_3c,
                (image_size, image_size),
                order=3,
                preserve_range=True,
                mode="constant",
                anti_aliasing=True,
            )

            resize_img_01 = (resize_img - resize_img.min()) / np.clip(
                resize_img.max() - resize_img.min(),
                a_min=1e-8,
                a_max=None,
            )

            out_file = (
                output_root
                / class_name
                / "imgs"
                / f"{prefix}_{volume_stem}-{str(i).zfill(3)}.npy"
            )

            np.save(out_file, resize_img_01.astype(np.float32))
            saved_count += 1

        return f"OK | {nii_file.name} | saved {saved_count} slices"

    except Exception as e:
        return f"ERROR | {nii_file} | {e}"


def main():
    dataset_prefix = "ACDC"
    modality = "MR"
    anatomy = "Cardiac"
    prefix = f"{dataset_prefix}_{modality}_{anatomy}"

    input_root = Path("./datasets/ACDC/images/ACDC_class_split")
    output_root = Path("./datasets/prep_datasets") / prefix

    image_size = 1024
    remove_empty_slices = True
    num_workers = max(1, os.cpu_count() - 1)

    for class_name in ["RV_class", "not_RV_class"]:
        (output_root / class_name / "imgs").mkdir(parents=True, exist_ok=True)

    nii_files = sorted(
        p for p in input_root.rglob("*.nii.gz")
        if "_gt" not in p.name and "4d" not in p.name.lower()
    )

    print(f"Found {len(nii_files)} image volumes.")
    print(f"Using {num_workers} worker processes.")

    futures = []
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        for nii_file in nii_files:
            futures.append(
                executor.submit(
                    process_one_volume,
                    str(nii_file),
                    str(input_root),
                    str(output_root),
                    prefix,
                    modality,
                    image_size,
                    remove_empty_slices,
                )
            )

        for future in tqdm(as_completed(futures), total=len(futures)):
            msg = future.result()
            if msg.startswith("ERROR") or msg.startswith("Could not") or msg.startswith("Unexpected"):
                print(msg)

    print("Done.")


if __name__ == "__main__":
    main()

Found 300 image volumes.
Using 95 worker processes.


  0%|          | 0/300 [00:00<?, ?it/s]WARNING: In /tmp/SimpleITK-build/ITK/Modules/IO/NIFTI/src/itkNiftiImageIO.cxx, line 2008
NiftiImageIO (0x55ac674dcb40): datasets/ACDC/images/ACDC_class_split/RV_class/patient090/patient090_frame11.nii.gz has unexpected scales in sform

NiftiImageIO (0x55ac674dcb40): datasets/ACDC/images/ACDC_class_split/RV_class/patient124/patient124_frame07.nii.gz has unexpected scales in sform

NiftiImageIO (0x55ac674dcb40): datasets/ACDC/images/ACDC_class_split/not_RV_class/patient003/patient003_frame15.nii.gz has unexpected scales in sform

NiftiImageIO (0x55ac674dcb40): datasets/ACDC/images/ACDC_class_split/not_RV_class/patient012/patient012_frame01.nii.gz has unexpected scales in sform

NiftiImageIO (0x55ac674dcb40): datasets/ACDC/images/ACDC_class_split/RV_class/patient094/patient094_frame07.nii.gz has unexpected scales in sform

NiftiImageIO (0x55ac674dcb40): datasets/ACDC/images/ACDC_class_split/RV_class/patient098/patient098_frame09.nii.gz has unexpected

Done.


## OASIS3 medSAM preprocessing

In [4]:
# -*- coding: utf-8 -*-
import os
from os.path import join
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import SimpleITK as sitk
from skimage import transform
from tqdm import tqdm


def ensure_3d(arr):
    """Ensure array has shape (Z, H, W)."""
    if arr.ndim == 2:
        arr = arr[None, :, :]
    elif arr.ndim != 3:
        raise ValueError(f"Expected 2D or 3D image, got shape {arr.shape}")
    return arr


def process_one_file(
    name,
    nii_path,
    npy_path,
    prefix,
    img_name_suffix,
    image_size,
    num_slices_to_keep,
    save_nifti,
    modality,
    window_level,
    window_width,
):
    image_name = name
    image_stem = name[:-len(img_name_suffix)]  # remove ".nii.gz"

    try:
        # Read volume
        img_sitk = sitk.ReadImage(join(nii_path, image_name))
        image_data = sitk.GetArrayFromImage(img_sitk)  # (Z, H, W)
        image_data = ensure_3d(image_data)

        # Preprocessing
        if modality == "CT":
            lower_bound = window_level - window_width / 2
            upper_bound = window_level + window_width / 2
            image_data_pre = np.clip(image_data, lower_bound, upper_bound)

            denom = np.max(image_data_pre) - np.min(image_data_pre)
            if denom <= 0:
                return ("skipped", name, "constant-valued CT volume after windowing")

            image_data_pre = (
                (image_data_pre - np.min(image_data_pre))
                / denom
                * 255.0
            )

        else:
            nonzero = image_data[image_data > 0]
            if nonzero.size == 0:
                return ("skipped", name, "all voxels are zero")

            lower_bound = np.percentile(nonzero, 0.5)
            upper_bound = np.percentile(nonzero, 99.5)
            image_data_pre = np.clip(image_data, lower_bound, upper_bound)

            denom = np.max(image_data_pre) - np.min(image_data_pre)
            if denom <= 0:
                return ("skipped", name, "constant-valued MR volume after clipping")

            image_data_pre = (
                (image_data_pre - np.min(image_data_pre))
                / denom
                * 255.0
            )
            image_data_pre[image_data == 0] = 0

        image_data_pre = np.uint8(image_data_pre)

        # Keep middle non-empty slices
        nonzero_slices = np.where(np.any(image_data_pre > 0, axis=(1, 2)))[0]
        if len(nonzero_slices) == 0:
            return ("skipped", name, "no non-empty slices after preprocessing")

        mid_idx = len(nonzero_slices) // 2
        start_idx = max(0, mid_idx - num_slices_to_keep // 2)
        end_idx = start_idx + num_slices_to_keep

        if end_idx > len(nonzero_slices):
            end_idx = len(nonzero_slices)
            start_idx = max(0, end_idx - num_slices_to_keep)

        z_index = nonzero_slices[start_idx:end_idx]
        img_roi = image_data_pre[z_index, :, :]

        # Save compressed volume
        np.savez_compressed(
            join(npy_path, prefix + image_stem + ".npz"),
            imgs=img_roi,
            spacing=np.array(img_sitk.GetSpacing()),
            original_shape=np.array(image_data.shape),
            kept_slices=z_index,
        )

        # Optional sanity-check NIfTI
        if save_nifti:
            img_roi_sitk = sitk.GetImageFromArray(img_roi)
            sitk.WriteImage(
                img_roi_sitk,
                join(npy_path, prefix + image_stem + "_img.nii.gz"),
            )

        # Save each slice as resized 3-channel .npy
        for i in range(img_roi.shape[0]):
            img_i = img_roi[i, :, :]
            img_3c = np.repeat(img_i[:, :, None], 3, axis=-1)

            resize_img = transform.resize(
                img_3c,
                (image_size, image_size),
                order=1,
                preserve_range=True,
                mode="constant",
                anti_aliasing=True,
            )

            resize_img_01 = (resize_img / 255.0).astype(np.float32)

            np.save(
                join(
                    npy_path,
                    "imgs",
                    prefix + image_stem + "-" + str(i).zfill(3) + ".npy",
                ),
                resize_img_01,
            )

        return ("done", name, img_roi.shape[0])

    except Exception as e:
        return ("failed", name, str(e))


if __name__ == "__main__":
    # =========================
    # Config
    # =========================
    dataset_prefix = "OASIS3"
    modality = "CT"   # "CT" or e.g. "MR_T1w"
    anatomy = "Brain"
    img_name_suffix = ".nii.gz"
    prefix = f"{dataset_prefix}_{modality}_{anatomy}_"

    nii_path = "datasets/oasis3_T2w_reformated/images"
    npy_path = "datasets/prep_datasets/" + prefix[:-1]

    os.makedirs(npy_path, exist_ok=True)
    os.makedirs(join(npy_path, "imgs"), exist_ok=True)

    image_size = 1024
    num_slices_to_keep = 10
    SAVE_NIFTI = False
    MAX_WORKERS = 24

    # Only used when modality == "CT"
    WINDOW_LEVEL = 40
    WINDOW_WIDTH = 400

    # =========================
    # Collect files
    # =========================
    names = sorted(
        [name for name in os.listdir(nii_path) if name.endswith(img_name_suffix)]
    )
    print(f"number of image files = {len(names)}")
    print(f"using MAX_WORKERS = {MAX_WORKERS}")

    # =========================
    # Parallel processing
    # =========================
    done_count = 0
    skipped_count = 0
    failed_count = 0

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(
                process_one_file,
                name,
                nii_path,
                npy_path,
                prefix,
                img_name_suffix,
                image_size,
                num_slices_to_keep,
                SAVE_NIFTI,
                modality,
                WINDOW_LEVEL,
                WINDOW_WIDTH,
            )
            for name in names
        ]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing volumes"):
            status, name, info = future.result()

            if status == "done":
                done_count += 1
            elif status == "skipped":
                skipped_count += 1
                print(f"Skipped {name}: {info}")
            elif status == "failed":
                failed_count += 1
                print(f"Failed {name}: {info}")

    print("\nFinished.")
    print(f"Done   : {done_count}")
    print(f"Skipped: {skipped_count}")
    print(f"Failed : {failed_count}")

number of image files = 4051
using MAX_WORKERS = 24


Processing volumes:   3%|▎         | 131/4051 [00:25<10:13,  6.39it/s]

Failed sub-OAS30033_ses-d0133_run-02_T2w.nii.gz: Expected 2D or 3D image, got shape (26, 60, 96, 96)


Processing volumes:  51%|█████     | 2049/4051 [05:37<04:57,  6.72it/s]

Failed sub-OAS30649_ses-d0098_acq-TSE_T2w.nii.gz: Expected 2D or 3D image, got shape (2, 128, 128, 128)


...

Failed sub-OAS30724_ses-d3115_run-02_T2w.nii.gz: Expected 2D or 3D image, got shape (26, 60, 96, 96)


Processing volumes:  65%|██████▌   | 2644/4051 [07:14<03:29,  6.70it/s]

Failed sub-OAS30815_ses-d0148_run-02_T2w.nii.gz: Expected 2D or 3D image, got shape (26, 60, 96, 96)


Processing volumes: 100%|██████████| 4051/4051 [11:18<00:00,  5.97it/s]



Finished.
Done   : 4047
Skipped: 0
Failed : 4


In [7]:
import os
from pathlib import Path
import shutil

src_dir = Path("datasets/prep_datasets/OASIS3_CT_Brain/imgs")
t2w_dir = Path("datasets/prep_datasets/OASIS3_T2w_Brain/imgs")

# create destination folder if it doesn't exist
t2w_dir.mkdir(parents=True, exist_ok=True)

moved = 0

for f in src_dir.iterdir():
    if not f.is_file():
        continue

    if "T2w" in f.name:
        dst = t2w_dir / f.name
        shutil.move(str(f), str(dst))
        moved += 1

print(f"Moved {moved} T2w files to {t2w_dir}")

Moved 0 T2w files to datasets/prep_datasets/OASIS3_T2w_Brain/imgs
